***INFERENCE AND SUBMISSION***

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.preprocessing import LabelEncoder
from scipy.ndimage import gaussian_filter1d

device = torch.device('cpu')
base_path = '/kaggle/input/competitions/birdclef-2026'

model_weights_asym = '/kaggle/input/datasets/adrianosemerano/birdclef/sed_final_phaseB_Asymmetric.pth' 
model_weights_resnet = '/kaggle/input/datasets/adrianosemerano/birdclef/sed_final_phaseB_SE-ResNet.pth' 

# Reconstruct LabelEncoder
train_csv = os.path.join(base_path, 'train.csv')
df = pd.read_csv(train_csv)
label_encoder = LabelEncoder()
label_encoder.fit(df['primary_label'])
num_classes = len(label_encoder.classes_)

class AudioToSpectrogramGPU(nn.Module):
    def __init__(self, sr=32000, n_mels=256, n_fft=2048, hop_length=512, f_min=40, f_max=15000):
        super(AudioToSpectrogramGPU, self).__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length, f_min=f_min, f_max=f_max, n_mels=n_mels, power=2.0 
        )
        self.eps, self.s, self.alpha, self.delta, self.r = 1e-6, 0.025, 0.98, 2.0, 0.5
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=24)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=64)

    def forward(self, waveform):
        x = self.mel_spec(waveform)
        ema = x.clone()
        for t in range(1, x.size(-1)):
            ema[..., t] = (1 - self.s) * ema[..., t - 1] + self.s * x[..., t]
        x = (x / (self.eps + ema)**self.alpha + self.delta)**self.r - self.delta**self.r
        x = (x - x.mean()) / (x.std() + 1e-6)
        if self.training:
            x = self.freq_mask(x)
            x = self.time_mask(x)
        return x

class AttentivePooling(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(AttentivePooling, self).__init__()
        self.attention = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)
        self.classifier = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)

    def forward(self, x):
        att_weights = torch.softmax(self.attention(x), dim=-1)
        frame_logits = self.classifier(x)
        clip_logits = torch.sum(att_weights * frame_logits, dim=-1)
        return clip_logits, frame_logits

class AsymmetricConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool_freq=2, pool_time=1):
        super(AsymmetricConvBlock, self).__init__()
        self.conv_f = nn.Conv2d(in_channels, out_channels, kernel_size=(5, 1), padding=(2, 0), bias=False)
        self.bn_f = nn.BatchNorm2d(out_channels)
        self.conv_t = nn.Conv2d(out_channels, out_channels, kernel_size=(1, 5), padding=(0, 2), bias=False)
        self.bn_t = nn.BatchNorm2d(out_channels)
        self.pool = nn.MaxPool2d(kernel_size=(pool_freq, pool_time))

    def forward(self, x):
        return self.pool(F.relu(self.bn_t(self.conv_t(F.relu(self.bn_f(self.conv_f(x)))))))

class BirdSED_AsymmetricCNN(nn.Module):
    def __init__(self, num_classes):
        super(BirdSED_AsymmetricCNN, self).__init__()
        self.audio_extractor = AudioToSpectrogramGPU()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        self.block1 = AsymmetricConvBlock(32, 64, pool_freq=2, pool_time=1)
        self.block2 = AsymmetricConvBlock(64, 128, pool_freq=2, pool_time=2)
        self.block3 = AsymmetricConvBlock(128, 256, pool_freq=2, pool_time=2)
        self.block4 = AsymmetricConvBlock(256, 512, pool_freq=2, pool_time=2)
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.dropout = nn.Dropout(0.5)
        self.sed_head = AttentivePooling(in_channels=512, num_classes=num_classes)

    def forward(self, waveform):
        x = self.stem(self.audio_extractor(waveform))
        x = self.block4(self.block3(self.block2(self.block1(x))))
        clip_logits, _ = self.sed_head(self.dropout(self.freq_pool(x).squeeze(2)))
        return clip_logits


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False), nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1).expand_as(x)

class SEResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(SEResNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)
        self.shortcut = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False), nn.BatchNorm2d(out_channels)) if stride != 1 or in_channels != out_channels else nn.Sequential()
    def forward(self, x):
        return F.relu(self.se(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))) + self.shortcut(x))

class BirdSED_SEResNet(nn.Module):
    def __init__(self, num_classes):
        super(BirdSED_SEResNet, self).__init__()
        self.audio_extractor = AudioToSpectrogramGPU()
        self.in_channels = 32
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, stride=1)
        self.layer2 = self._make_layer(128, stride=2)
        self.layer3 = self._make_layer(256, stride=2)
        self.layer4 = self._make_layer(512, stride=2)
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.dropout = nn.Dropout(0.5)
        self.sed_head = AttentivePooling(in_channels=512, num_classes=num_classes)

    def _make_layer(self, out_channels, stride):
        layer = SEResNetBlock(self.in_channels, out_channels, stride)
        self.in_channels = out_channels
        return layer

    def forward(self, waveform):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.pool1(F.relu(self.bn1(self.conv1(self.audio_extractor(waveform)))))))))
        clip_logits, _ = self.sed_head(self.dropout(self.freq_pool(x).squeeze(2)))
        return clip_logits


def get_padded_chunk(wf, start, end, chunk_samples):
    c_len = end - start
    pad_left = max(0, -start)
    start_safe = max(0, start)
    end_safe = min(end, wf.shape[1])
    pad_right = max(0, end - wf.shape[1])
    chunk = wf[:, start_safe:end_safe]
    if pad_left > 0 or pad_right > 0:
        chunk = torch.nn.functional.pad(chunk, (pad_left, pad_right))
    return chunk

def apply_temporal_smoothing(probs_matrix, sigma=1.0):
    smoothed_probs = np.zeros_like(probs_matrix)
    for class_idx in range(probs_matrix.shape[1]):
        smoothed_probs[:, class_idx] = gaussian_filter1d(probs_matrix[:, class_idx], sigma=sigma)
    return smoothed_probs

model_asym = BirdSED_AsymmetricCNN(num_classes=num_classes)
model_asym.load_state_dict(torch.load(model_weights_asym, map_location=device))
model_asym.to(device).eval()

model_resnet = BirdSED_SEResNet(num_classes=num_classes)
model_resnet.load_state_dict(torch.load(model_weights_resnet, map_location=device))
model_resnet.to(device).eval()

sample_sub_path = os.path.join(base_path, 'sample_submission.csv')
sample_df = pd.read_csv(sample_sub_path)
expected_species_columns = sample_df.columns[1:] 
test_audio_dir = os.path.join(base_path, 'test_soundscapes')

predictions_dict = {}
predicted_species_names = label_encoder.inverse_transform(range(num_classes))
species_to_idx = {sp: idx for idx, sp in enumerate(predicted_species_names)}

if os.path.exists(test_audio_dir) and len([f for f in os.listdir(test_audio_dir) if f.endswith('.ogg')]) > 0:
    test_files = [f for f in os.listdir(test_audio_dir) if f.endswith('.ogg')]
    print(f"Found {len(test_files)} hidden test files. Processing dynamically on CPU with TTA Ensemble...")
    
    for file in test_files:
        file_path = os.path.join(test_audio_dir, file)
        file_stem = file.replace('.ogg', '')
        
        try:
            waveform, sr = torchaudio.load(file_path)
            if waveform.shape[0] > 1: 
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            chunk_samples = 5 * 32000
            shift_samples = int(0.5 * 32000) 
            total_samples = waveform.shape[1]
            num_chunks = total_samples // chunk_samples 
            
            file_probs = []
            batch_size = 16 
            
            for b in range(0, num_chunks, batch_size):
                wave_chunks_base, wave_chunks_left, wave_chunks_right = [], [], []
                
                for i in range(b, min(b + batch_size, num_chunks)):
                    start = i * chunk_samples
                    end = start + chunk_samples
                    wave_chunks_base.append(get_padded_chunk(waveform, start, end, chunk_samples))
                    wave_chunks_left.append(get_padded_chunk(waveform, start - shift_samples, end - shift_samples, chunk_samples))
                    wave_chunks_right.append(get_padded_chunk(waveform, start + shift_samples, end + shift_samples, chunk_samples))
                    
                batch_base = torch.stack(wave_chunks_base).to(device)
                batch_left = torch.stack(wave_chunks_left).to(device)
                batch_right = torch.stack(wave_chunks_right).to(device)
                
                with torch.no_grad():
                    # Asymmetric CNN Predictions
                    a_base = torch.sigmoid(model_asym(batch_base)).cpu().numpy()
                    a_left = torch.sigmoid(model_asym(batch_left)).cpu().numpy()
                    a_right = torch.sigmoid(model_asym(batch_right)).cpu().numpy()
                    probs_asym_tta = (a_base + a_left + a_right) / 3.0
                    
                    # SE-ResNet Predictions
                    r_base = torch.sigmoid(model_resnet(batch_base)).cpu().numpy()
                    r_left = torch.sigmoid(model_resnet(batch_left)).cpu().numpy()
                    r_right = torch.sigmoid(model_resnet(batch_right)).cpu().numpy()
                    probs_resnet_tta = (r_base + r_left + r_right) / 3.0
                    
                    # ENSEMBLE: Average the models
                    probs_ensemble = (probs_asym_tta + probs_resnet_tta) / 2.0
                    file_probs.append(probs_ensemble)
            
            if len(file_probs) > 0:
                file_probs = np.vstack(file_probs)
                smoothed_probs = apply_temporal_smoothing(file_probs, sigma=1.0)
                
                for i in range(num_chunks):
                    end_second = (i + 1) * 5
                    row_id = f"{file_stem}_{end_second}"
                    predictions_dict[row_id] = smoothed_probs[i] 
                    
            del waveform
            gc.collect()
            
        except Exception as e:
            print(f"Error processing {file}: {e}")

for idx, row in sample_df.iterrows():
    r_id = row['row_id']
    if r_id in predictions_dict:
        pred_probs = predictions_dict[r_id]
        for col in expected_species_columns:
            if col in species_to_idx:
                sample_df.at[idx, col] = pred_probs[species_to_idx[col]]

sample_df.to_csv('submission.csv', index=False)
print("Submission.csv generated")